In [ ]:


### **Question 1: What is Ensemble Learning in machine learning? Explain the key idea behind it.**

Ensemble learning is a machine learning paradigm where multiple models (often called "base models" or "weak learners") are trained to solve the same problem and are then combined to produce a single final prediction. The key idea behind it is the "wisdom of the crowd": by aggregating the predictions of multiple diverse models, the errors made by a single model can be compensated for by the others. This leads to a final model that is generally more robust, accurate, and stable than any individual constituent model, effectively reducing variance, bias, or both.

### **Question 2: What is the difference between Bagging and Boosting?**

* **Bagging (Bootstrap Aggregating):** Aims to reduce **variance** and prevent overfitting. It trains multiple base models independently and in parallel on different random subsets of the training data (created with replacement). The final prediction is made by averaging the outputs (for regression) or taking a majority vote (for classification). Random Forest is a popular example.
* **Boosting:** Aims to reduce **bias** and improve predictive accuracy. It trains base models sequentially, where each new model is focused on correcting the errors (misclassifications or residuals) made by the previous models. The final prediction is a weighted sum of all the models. Gradient Boosting and AdaBoost are popular examples.

### **Question 3: What is bootstrap sampling and what role does it play in Bagging methods like Random Forest?**

Bootstrap sampling is a statistical technique that involves drawing random samples from a dataset *with replacement*. This means the same data point can appear multiple times in a single sample.
In Bagging methods like Random Forest, bootstrap sampling is used to create a unique training subset for each individual decision tree. This role is crucial because it injects randomness and diversity into the ensemble. Since each tree trains on slightly different data, they make different errors. When aggregated, these errors cancel out, reducing the overall variance of the model and preventing overfitting.

### **Question 4: What are Out-of-Bag (OOB) samples and how is OOB score used to evaluate ensemble models?**

Out-of-Bag (OOB) samples are the data points from the original dataset that are *not* selected during the bootstrap sampling process for a particular base model. Because sampling is done with replacement, roughly 36.8% of the data is left out for each tree.
The OOB score is calculated by passing these unseen OOB data points through the specific trees that were not trained on them, and then aggregating those predictions. This score acts as a built-in cross-validation metric, allowing you to evaluate the model's generalization performance on unseen data without needing to split off a separate validation set.

### **Question 5: Compare feature importance analysis in a single Decision Tree vs. a Random Forest.**

* **Single Decision Tree:** Feature importance is calculated by looking at how much a specific feature decreases the impurity (e.g., Gini impurity or Entropy) across all the nodes where it is used to split the data. However, in a single tree, this measure can be highly unstable and prone to overfitting, as the tree might lock onto a specific noise pattern in the training data.
* **Random Forest:** Feature importance is calculated by averaging the importance of each feature across all the trees in the forest. Because Random Forests use both bootstrap sampling (different data for each tree) and feature randomness (different subsets of features evaluated at each split), the aggregated feature importance is much more robust, reliable, and less sensitive to noise or specific data quirks compared to a single tree.

---

### **Question 6: Write a Python program to load Breast Cancer, train RF, print top 5 features.**

```python
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.ensemble import RandomForestClassifier

# Load the Breast Cancer dataset
data = load_breast_cancer()
X, y = data.data, data.target

# Train a Random Forest Classifier
rf_classifier = RandomForestClassifier(random_state=42)
rf_classifier.fit(X, y)

# Extract feature importances
importances = pd.Series(rf_classifier.feature_importances_, index=data.feature_names)

# Print the top 5 most important features
top_5_features = importances.sort_values(ascending=False).head(5)
print("Top 5 most important features:")
print(top_5_features)

```

---

### **Question 7: Write a Python program to Train a Bagging Classifier on Iris, evaluate accuracy and compare with a single Decision Tree.**

```python
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import BaggingClassifier
from sklearn.metrics import accuracy_score

# Load the Iris dataset
iris = load_iris()
X_train, X_test, y_train, y_test = train_test_split(iris.data, iris.target, test_size=0.3, random_state=42)

# Train and evaluate a single Decision Tree
dt = DecisionTreeClassifier(random_state=42)
dt.fit(X_train, y_train)
dt_pred = dt.predict(X_test)
dt_accuracy = accuracy_score(y_test, dt_pred)

# Train and evaluate a Bagging Classifier using Decision Trees
bagging = BaggingClassifier(estimator=DecisionTreeClassifier(random_state=42),
                            n_estimators=50, random_state=42)
bagging.fit(X_train, y_train)
bagging_pred = bagging.predict(X_test)
bagging_accuracy = accuracy_score(y_test, bagging_pred)

# Compare results
print(f"Single Decision Tree Accuracy: {dt_accuracy:.4f}")
print(f"Bagging Classifier Accuracy: {bagging_accuracy:.4f}")

```

---

### **Question 8: Write a Python program to tune Random Forest hyperparameters using GridSearchCV.**

```python
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# Load dataset and split
data = load_breast_cancer()
X_train, X_test, y_train, y_test = train_test_split(data.data, data.target, test_size=0.3, random_state=42)

# Define the model and hyperparameter grid
rf = RandomForestClassifier(random_state=42)
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [None, 10, 20, 30]
}

# Set up GridSearchCV
grid_search = GridSearchCV(estimator=rf, param_grid=param_grid, cv=5, scoring='accuracy', n_jobs=-1)

# Train model with grid search
grid_search.fit(X_train, y_train)

# Evaluate the best model on test data
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)
final_accuracy = accuracy_score(y_test, y_pred)

# Print results
print(f"Best Parameters: {grid_search.best_params_}")
print(f"Final Accuracy: {final_accuracy:.4f}")

```

---

### **Question 9: Write a Python program to train a Bagging Regressor and RF Regressor on California Housing, compare MSE.**

```python
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.ensemble import BaggingRegressor, RandomForestRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error

# Load dataset
california = fetch_california_housing()
X_train, X_test, y_train, y_test = train_test_split(california.data, california.target, test_size=0.3, random_state=42)

# Train and evaluate Bagging Regressor
bagging_reg = BaggingRegressor(estimator=DecisionTreeRegressor(random_state=42),
                               n_estimators=50, random_state=42)
bagging_reg.fit(X_train, y_train)
bagging_pred = bagging_reg.predict(X_test)
bagging_mse = mean_squared_error(y_test, bagging_pred)

# Train and evaluate Random Forest Regressor
rf_reg = RandomForestRegressor(n_estimators=50, random_state=42)
rf_reg.fit(X_train, y_train)
rf_pred = rf_reg.predict(X_test)
rf_mse = mean_squared_error(y_test, rf_pred)

# Compare MSE (Lower is better)
print(f"Bagging Regressor MSE: {bagging_mse:.4f}")
print(f"Random Forest Regressor MSE: {rf_mse:.4f}")

```

---

### **Question 10: Step-by-step approach to predict loan default using ensemble techniques.**

* **Choose between Bagging or Boosting:** I would start with **Boosting** (like XGBoost or LightGBM). While Bagging (Random Forest) is an excellent robust baseline, tabular financial data involving transaction history often contains complex, non-linear patterns with subtle margins of error. Boosting excels here because it sequentially focuses on the hardest-to-predict customers (the errors), generally yielding the highest predictive accuracy and ROC-AUC for credit scoring.
* **Handle overfitting:** Since Boosting is prone to overfitting if unchecked, I would control it by tuning hyperparameters: lowering the learning rate (shrinkage), limiting the maximum tree depth (keeping trees shallow, e.g., 3-6), applying L1/L2 regularization, and using early stopping during training so the model halts when validation performance stops improving.
* **Select base models:** I would select **Decision Trees** as the base models. They naturally handle mixed data types (categorical demographics and numerical transaction histories), ignore missing values gracefully, and do not require extensive feature scaling, which is ideal for raw financial data.
* **Evaluate performance using cross-validation:** I would use **Stratified K-Fold Cross-Validation** (typically 5 or 10 folds). Loan default data is inherently imbalanced (there are far more non-defaulters than defaulters). Stratification ensures every fold maintains the same ratio of defaults to non-defaults. I would evaluate using metrics like Precision-Recall AUC and F1-score rather than simple accuracy, which is misleading on imbalanced datasets.
* **Justify how ensemble learning improves decision-making:** In banking, a single decision tree might overfit a specific niche of clients or completely miss a nuanced risk pattern. By using an ensemble, the institution combines hundreds of slightly different models into one robust risk score. This reduces false positives (unfairly denying loans to good customers) and false negatives (approving loans to high-risk defaulters). It ultimately leads to safer, fairer, and more profitable lending decisions compared to traditional linear models.